## Cross-filtering in dash

### Notebook preparation

Let's start by installing and importing the necessary libraries

In [19]:
!pip install dash

In [20]:
import dash
import json
import pandas as pd
import numpy as np
import plotly.express as px
import threading
from dash import dcc, html, Input, Output
from google.colab import output, drive

In [21]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Creating the visualization

#### Supporting solutions

Preparing the needed data

In [22]:
population = pd.read_csv('/content/drive/My Drive/Colab Notebooks/Vis/population.csv')
new_books = pd.read_csv("/content/drive/My Drive/Colab Notebooks/Vis/new-books-per-million.csv")
new_books = new_books[(new_books["Year"]>=1940) & (new_books["Year"]<=1996)]
selected_books = new_books[new_books["Year"]==1996][["Entity", "Book titles per capita (Fink-Jensen 2015)"]]
selected_pops = population[population["Year"]==1996][["Entity", "Population (historical estimates)"]]
data = pd.merge(selected_books, selected_pops, how="inner", on="Entity")

Creacting the chart used in the application

In [23]:
scatter_figure = px.scatter(data_frame=data,
                 x="Book titles per capita (Fink-Jensen 2015)",
                 y="Population (historical estimates)",
                 text="Entity",
                 log_x=True,
                 log_y=True,
                 template="none",
                 title="Country population vs number of new book titles per capita in 1996")
scatter_figure.update_traces(textposition="bottom right")

Below, we have the code of the first application used in the **dash crossfiltering** presentation.

In [24]:
app = dash.Dash()

app.layout = html.Div(
    children=[
        # id sjednotíme na div-value
        html.P(id="div-value"),
        dcc.Graph(id="scatter", figure=scatter_figure),
    ]
)

@app.callback(
    Output("div-value", "children"),      # stejné id jako v layoutu
    Input("scatter", "clickData")        # místo hoverData používáme clickData
)
def show_me(data):
    # data může být na začátku None, tak ať to nespadne
    if data is None:
        return "Click on a point in the scatter plot."
    return json.dumps(data)

Below, we have the code of the second application used in the **dash crossfiltering** presentation.

In [25]:
app = dash.Dash()

app.layout = html.Div(
    children=[
        dcc.Graph(
            id="scatter",
            figure=scatter_figure
        ),
        dcc.Graph(id="line")
    ]
)


@app.callback(
    Output("line", "figure"),
    Input("scatter", "clickData")        # ⬅ tady je ta hlavní změna
)
def build_line_chart(clickData):
    """
    Vezme informaci o naposledy kliknutém bodu (clickData),
    zjistí vybranou zemi a zobrazí její časovou řadu v line chartu.
    """

    # 1) Když ještě nikdo neklikl, ukážeme prázdný graf nebo default
    if clickData is None:
        # defaultně třeba první zemi v datech
        default_country = new_books["Entity"].iloc[0]
        selected_data = new_books[new_books["Entity"] == default_country]

        fig = px.line(
            data_frame=selected_data,
            x="Year",
            y="Book titles per capita (Fink-Jensen 2015)",
            template="none",
            title=f"Book titles per capita in %s for years 1948-1996" % default_country,
        )
        return fig

    # 2) Když se kliklo: vybereme z 'clickData' název země
    selected_country = clickData["points"][0]["text"]
    selected_data = new_books[new_books["Entity"] == selected_country]

    # 3) Vykreslíme line chart pro vybranou zemi
    fig = px.line(
        data_frame=selected_data,
        x="Year",
        y="Book titles per capita (Fink-Jensen 2015)",
        template="none",
        title=f"Book titles per capita in %s for years 1948-1996" % selected_country,
    )

    return fig

### Displaying the visualization

To build the visualization using the code above, we also need to run a server and go to the application page.


In [26]:
from google.colab import output

PORT = 8060  # nebo 8050, pokud není obsazený

thread = threading.Thread(
    target=app.run,
    kwargs={"port": PORT, "debug": False, "use_reloader": False}
)
thread.start()

output.serve_kernel_port_as_iframe(port=PORT, height=600)

print("\n--- Souhrn ---")
print(f"1) Aplikace běží na portu {PORT}.")
print("2) V buňce nad tímto textem vidíš iframe s aplikací.")
print("3) Line chart se mění podle bodu, na který klikneš ve scatter grafu.")

Dash is running on http://127.0.0.1:8060/



<IPython.core.display.Javascript object>

INFO:dash.dash:Dash is running on http://127.0.0.1:8060/




--- Souhrn ---
1) Aplikace běží na portu 8060.
2) V buňce nad tímto textem vidíš iframe s aplikací.
3) Line chart se mění podle bodu, na který klikneš ve scatter grafu.


In [28]:
from google.colab import output
import threading

PORT = 8050  # můžeš změnit na 8060, kdyby byl 8050 obsazený

def run_app():
    app.run(port=PORT, debug=False, use_reloader=False)

thread = threading.Thread(target=run_app)
thread.start()

# otevře se nové okno / záložka s adresou přes proxy (vypadá jako localhost)
output.serve_kernel_port_as_window(port=PORT)

print("\n--- Souhrn ---")
print(f"1) Dash aplikace běží na portu {PORT}.")
print("2) Nad tímto textem se objeví odkaz / okno s appkou.")
print("3) Line chart se mění podle bodu, na který klikneš ve scatter grafu.")

Dash is running on http://127.0.0.1:8050/

Try `serve_kernel_port_as_iframe` instead. 


INFO:dash.dash:Dash is running on http://127.0.0.1:8050/



<IPython.core.display.Javascript object>

 * Serving Flask app '__main__'

--- Souhrn ---
1) Dash aplikace běží na portu 8050.
2) Nad tímto textem se objeví odkaz / okno s appkou.
3) Line chart se mění podle bodu, na který klikneš ve scatter grafu.
 * Debug mode: off


Address already in use
Port 8050 is in use by another program. Either identify and stop that program, or start the server with a different port.


Stručná interpretace (co se změnilo)

Původně:

scatter graf posílal do callbacku hoverData – informace o posledním místě, nad kterým se nacházel kurzor,

to znamenalo, že line chart reagoval už jen na najetí myší.

Po úpravě:

používáme clickData,

line chart se aktualizuje jen tehdy, když na bod klikneš,

uživatel tak má větší kontrolu nad tím, co se zobrazí (pohyb myši graf nemění).

In [29]:
# --- 1) Scatter graf pro rok 1996 (kratší, ne přes celou obrazovku) ---

scatter_figure = px.scatter(
    data_frame=data,
    x="Book titles per capita (Fink-Jensen 2015)",
    y="Population (historical estimates)",
    text="Entity",
    log_x=True,
    log_y=True,
    template="none",
    title="Country population vs number of new book titles per capita in 1996",
)

scatter_figure.update_traces(textposition="bottom right")

# zmenšíme graf (šířka/výška v pixelech)
scatter_figure.update_layout(
    width=900,
    height=450,
    margin=dict(l=40, r=40, t=80, b=40),
)


# --- 2) Dash aplikace: scatter (nahoře) + line chart (dole) ---

app = dash.Dash()

app.layout = html.Div(
    children=[
        html.H2("Cross-filtering of books and population"),

        # SCATTER GRAF – výběr země kliknutím na bod
        dcc.Graph(
            id="scatter",
            figure=scatter_figure,
            style={"width": "900px", "height": "450px"},
        ),

        # LINE GRAF – časová řada pro vybranou zemi
        dcc.Graph(
            id="line",
            style={"width": "900px", "height": "450px"},
        ),
    ],
    style={
        "display": "flex",
        "flexDirection": "column",
        "alignItems": "center",
        "fontFamily": "Verdana",
    },
)


# --- 3) Callback: místo hoverData používáme clickData ---

@app.callback(
    Output("line", "figure"),
    Input("scatter", "clickData"),   # <<< tady je ta hlavní změna
)
def build_line_chart(clickData):
    """
    Vezme informaci o naposledy kliknutém bodu (clickData),
    zjistí vybranou zemi a zobrazí její časovou řadu v line chartu.
    """

    # 1) Když ještě nikdo neklikl -> defaultní země (např. první v datasetu)
    if clickData is None:
        default_country = new_books["Entity"].iloc[0]
        selected_data = new_books[new_books["Entity"] == default_country]

        fig = px.line(
            data_frame=selected_data,
            x="Year",
            y="Book titles per capita (Fink-Jensen 2015)",
            template="none",
            title=f"Book titles per capita in {default_country} for years 1948–1996",
        )

        fig.update_layout(width=900, height=450)
        return fig

    # 2) Když se kliklo: vybereme z clickData název země
    selected_country = clickData["points"][0]["text"]
    selected_data = new_books[new_books["Entity"] == selected_country]

    # 3) Vykreslíme line graf pro vybranou zemi
    fig = px.line(
        data_frame=selected_data,
        x="Year",
        y="Book titles per capita (Fink-Jensen 2015)",
        template="none",
        title=f"Book titles per capita in {selected_country} for years 1948–1996",
    )

    fig.update_layout(width=900, height=450)
    return fig


# --- 4) Spuštění appky v Colabu + iframe + localhost odkaz ve výpisu ---

from google.colab import output
import threading

PORT = 8060  # pokud by byl obsazený, změň třeba na 8061

def run_app():
    app.run(port=PORT, debug=False, use_reloader=False)

thread = threading.Thread(target=run_app)
thread.start()

# okno s aplikací přímo v buňce
output.serve_kernel_port_as_iframe(port=PORT, height=900)

print("\n--- Výsledek / Souhrn ---")
print(f"1) Dash aplikace běží na adrese: http://127.0.0.1:{PORT}/")
print("2) Nad tímto textem vidíš oba grafy (scatter nahoře, line graf dole).")
print("3) Zemi NEvybíráš v dropdownu, ale kliknutím na bod ve scatter grafu.")
print("4) Line graf dole se vždy přepočítá podle naposledy kliknuté země.")


Dash is running on http://127.0.0.1:8060/



INFO:dash.dash:Dash is running on http://127.0.0.1:8060/



<IPython.core.display.Javascript object>


--- Výsledek / Souhrn ---
1) Dash aplikace běží na adrese: http://127.0.0.1:8060/
2) Nad tímto textem vidíš oba grafy (scatter nahoře, line graf dole).
3) Zemi NEvybíráš v dropdownu, ale kliknutím na bod ve scatter grafu.
4) Line graf dole se vždy přepočítá podle naposledy kliknuté země.
 * Serving Flask app '__main__'
 * Debug mode: off


In [30]:
from google.colab import output
import threading

PORT = 8050  # standardní Dash port

def run_app():
    # app je tvoje Dash instance z layoutu výše
    app.run(port=PORT, debug=False, use_reloader=False)

# spustíme server v samostatném vlákně
thread = threading.Thread(target=run_app)
thread.start()

# 🔹 TADY Colab vytvoří speciální URL (často ve tvaru https://localhost:8050/…)
app_url = output.serve_kernel_port_as_window(port=PORT)

print("\n--- Výsledek / odkaz ---")
print("Dash app běží a měla by se otevřít v novém okně.")
print("Pokud se neotevře automaticky, použij tento odkaz (kliknutím v notebooku):")
print(app_url)


Dash is running on http://127.0.0.1:8050/

Try `serve_kernel_port_as_iframe` instead. 


INFO:dash.dash:Dash is running on http://127.0.0.1:8050/



<IPython.core.display.Javascript object>


--- Výsledek / odkaz --- * Serving Flask app '__main__'

Dash app běží a měla by se otevřít v novém okně.
Pokud se neotevře automaticky, použij tento odkaz (kliknutím v notebooku):
None
 * Debug mode: off


Address already in use
Port 8050 is in use by another program. Either identify and stop that program, or start the server with a different port.


Změňte způsob fungování prezentace tak, aby se místo informace o tom, kde se nacházel kurzor, používala informace o tom, který prvek byl naposledy zvětšen.
Zvažte, proč nestačí změnit název nemovitosti.

In [31]:
population, new_books, data   # načteno
scatter_figure                # vytvořený pro rok 1996

In [35]:
scatter_figure.update_layout(
    width=1200,
    height=500
  ,
    margin=dict(l=40, r=40, t=80, b=40),
)

In [33]:
# ============================================================
#  EXERCISE 2 – crossfilter podle "last zoomed element"
# ============================================================

import numpy as np
import dash
from dash import html, dcc
from dash.dependencies import Input, Output

app2 = dash.Dash()

app2.layout = html.Div(
    children=[
        html.H2("Cross-filtering (Exercise 2 – zoom)"),

        # SCATTER GRAF – uživatel může zoomovat
        dcc.Graph(
            id="scatter-zoom",
            figure=scatter_figure,
            style={"width": "900px", "height": "450px"},
        ),

        # LINE GRAF – časová řada pro "poslední zazoomovaný" prvek
        dcc.Graph(
            id="line-zoom",
            style={"width": "900px", "height": "450px"},
        ),
    ],
    style={
        "display": "flex",
        "flexDirection": "column",
        "alignItems": "center",
        "fontFamily": "Verdana",
    },
)


@app2.callback(
    Output("line-zoom", "figure"),
    Input("scatter-zoom", "relayoutData"),   # !!! změna – reagujeme na ZOOM
)
def build_line_chart_zoom(relayoutData):
    """
    Vybere "poslední zoomovaný" prvek:
    - vezmeme rozsah x a y po zoomu,
    - spočítáme střed zoomované oblasti,
    - najdeme bod (zemi), který je tomuto středu nejblíž,
    - pro tuto zemi vykreslíme časovou řadu.
    """

    # 1) Pokud žádný zoom neproběhl -> použijeme defaultní zemi
    if not relayoutData or "xaxis.range[0]" not in relayoutData:
        default_country = new_books["Entity"].iloc[0]
        selected_data = new_books[new_books["Entity"] == default_country]

        fig = px.line(
            data_frame=selected_data,
            x="Year",
            y="Book titles per capita (Fink-Jensen 2015)",
            template="none",
            title=f"Book titles per capita in {default_country} for years 1948–1996",
        )
        fig.update_layout(width=900, height=450)
        return fig

    # 2) Přečteme rozsahy po zoomu
    x0 = relayoutData["xaxis.range[0]"]
    x1 = relayoutData["xaxis.range[1]"]
    y0 = relayoutData["yaxis.range[0]"]
    y1 = relayoutData["yaxis.range[1]"]

    # střed zoomu (na log škále to není dokonalé, ale pro účel cvičení stačí)
    cx = (x0 + x1) / 2
    cy = (y0 + y1) / 2

    # 3) Najdeme bod, který je středu zoomu nejblíž
    #    – pracujeme s daty použitými pro scatter_figure (df = data)
    df = data.copy()
    # vzdálenost ve 2D
    dist = np.sqrt((np.log10(df["Book titles per capita (Fink-Jensen 2015)"]) - cx) ** 2 +
                   (np.log10(df["Population (historical estimates)"]) - cy) ** 2)
    idx = dist.idxmin()
    selected_country = df.loc[idx, "Entity"]

    # 4) Vykreslíme line graf pro "zazoomovanou" zemi
    selected_data = new_books[new_books["Entity"] == selected_country]

    fig = px.line(
        data_frame=selected_data,
        x="Year",
        y="Book titles per capita (Fink-Jensen 2015)",
        template="none",
        title=f"Book titles per capita in {selected_country} for years 1948–1996 "
              "(selected by zoom)",
    )
    fig.update_layout(width=900, height=450)
    return fig


In [34]:
from google.colab import output
import threading

PORT2 = 8062  # jiný port než v exercise 1

def run_app2():
    app2.run(port=PORT2, debug=False, use_reloader=False)

thread = threading.Thread(target=run_app2)
thread.start()

app2_url = output.serve_kernel_port_as_window(port=PORT2)

print("\n--- Exercise 2 – výsledek ---")
print(f"Appka běží na adrese (Colab URL): {app2_url}")
print("1) Horní scatter graf – zazoomuj na oblast kolem nějaké země (např. USA).")
print("2) Callback vezme střed zoomu a vybere zemi, která je mu nejblíž.")
print("3) Spodní line graf ukáže časovou řadu pro tuto 'zazoomovanou' zemi.")
print("4) Nestačí změnit popisek – museli jsme nahradit hoverData vstupem relayoutData a přepsat logiku.")

Dash is running on http://127.0.0.1:8062/

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

INFO:dash.dash:Dash is running on http://127.0.0.1:8062/




--- Exercise 2 – výsledek ---
Appka běží na adrese (Colab URL): None
1) Horní scatter graf – zazoomuj na oblast kolem nějaké země (např. USA).
2) Callback vezme střed zoomu a vybere zemi, která je mu nejblíž.
3) Spodní line graf ukáže časovou řadu pro tuto 'zazoomovanou' zemi.
4) Nestačí změnit popisek – museli jsme nahradit hoverData vstupem relayoutData a přepsat logiku.
 * Serving Flask app '__main__'
 * Debug mode: off
